In [ ]:
#| default_exp send_alert


# Step 7: Slack notifications

![Step 7 diagram](https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/docs/diagrams/07-step.png)

**Goal:** every time we save a sighting, post a message to Slack.

We use a Slack **incoming webhook** — just a URL that accepts JSON. If you don't have a webhook set yet, the code falls back to a *dry-run* that prints the message instead.

Two functions live in `bird_watcher/send_alert.py`:

- `build_sighting_alert` — build the Slack Block Kit JSON
- `send_alert_to_slack` — POST it (or dry-run)

## Step 7.0 — Setup

In [ ]:
from pathlib import Path

import yaml

ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (root for root in ROOT_CANDIDATES if (root / "tutorials").exists()),
    Path.cwd(),
)
CONFIG_FILE = PROJECT_ROOT / "config.yaml"

CONFIG = {}
if CONFIG_FILE.exists():
    CONFIG = yaml.safe_load(CONFIG_FILE.read_text()) or {}
    if not isinstance(CONFIG, dict):
        raise TypeError(f"{CONFIG_FILE} must contain a top-level mapping")
else:
    print(f"Config file not found at {CONFIG_FILE}; using defaults.")

# Folder + file constants. _FOLDER = a directory, _FILE = a single file.
TUTORIALS_FOLDER = PROJECT_ROOT / "tutorials"
DATA_FOLDER = PROJECT_ROOT / "data"
SNAPSHOT_FOLDER = DATA_FOLDER / "snapshots"
CROP_FOLDER = DATA_FOLDER / "crops"
DB_FILE = DATA_FOLDER / "birds.db"
SAMPLE_BIRD_FILE = DATA_FOLDER / "samples" / "sample-bird.jpg"
MODEL_FILE = PROJECT_ROOT / "yolov8n.pt"

# From config.yaml (or hardcoded fallback)
PHONE_IP = str(CONFIG.get("phone_ip", "192.168.1.42"))
PHONE_URL = f"http://{PHONE_IP}:8080/photo.jpg"
SLACK_WEBHOOK = str(CONFIG.get("slack_webhook", ""))
HUGGINGFACE_API_KEY = str(CONFIG.get("huggingface_api_key", ""))

SNAPSHOT_FOLDER.mkdir(parents=True, exist_ok=True)
CROP_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Snapshot folder: {SNAPSHOT_FOLDER}")
print(f"Config file: {CONFIG_FILE}")
print(f"Phone URL: {PHONE_URL}")

from bird_watcher.save_sighting import save_sighting_to_db
print(f"Slack webhook set: {bool(SLACK_WEBHOOK)}")

## Step 7.1 — Slack messages use Block Kit

Slack Block Kit is JSON that describes the message layout. A header + a section with fields is enough for a sighting alert.

In [ ]:
payload = {
    "blocks": [
        {
            "type": "header",
            "text": {"type": "plain_text", "text": ":bird: New sighting — Northern Cardinal"},
        },
        {
            "type": "section",
            "fields": [
                {"type": "mrkdwn", "text": "*Species*\nNorthern Cardinal"},
                {"type": "mrkdwn", "text": "*Confidence*\n92%"},
            ],
        },
    ]
}
import json
print(json.dumps(payload, indent=2))

## Step 7.2 — Wrap the payload builder as `build_sighting_alert`

In [ ]:
#| export
from pathlib import Path

def build_sighting_alert(
    species: str,
    confidence: float,
    snapshot_file: Path,
    sighting_id: int | None = None,
) -> dict:
    """Build a Slack Block Kit message payload for one sighting.

    Args:
        species: the bird's name.
        confidence: 0.0 - 1.0.
        snapshot_file: the photo to attach.
        sighting_id: optional database id for cross-referencing.

    Returns:
        A dict ready to be POSTed as JSON to a Slack webhook.
    """
    confidence_pct = int(confidence * 100)
    header_text = f":bird: New sighting — {species}"
    if sighting_id is not None:
        header_text += f" (#{sighting_id})"
    return {
        "blocks": [
            {
                "type": "header",
                "text": {"type": "plain_text", "text": header_text},
            },
            {
                "type": "section",
                "fields": [
                    {"type": "mrkdwn", "text": f"*Species*\n{species}"},
                    {"type": "mrkdwn", "text": f"*Confidence*\n{confidence_pct}%"},
                    {"type": "mrkdwn", "text": f"*Photo*\n`{snapshot_file.name}`"},
                ],
            },
        ]
    }


## Step 7.3 — Add `send_alert_to_slack`

In [ ]:
#| export
import json
import os
import requests

def send_alert_to_slack(
    payload: dict,
    webhook_url: str | None = None,
    verbose: bool = True,
) -> bool:
    """POST a message payload to Slack. Prints locally if no webhook.

    Args:
        payload: the dict returned by `build_sighting_alert`.
        webhook_url: the Slack incoming webhook URL. If empty, this runs in
            dry-run mode and prints the payload instead.
        verbose: if True, print whether we sent or just previewed.

    Returns:
        True if the message was sent (or dry-run previewed), False if failed.
    """
    import json
    import requests

    url = webhook_url or ""
    if not url:
        if verbose:
            print("[dry-run] No SLACK_WEBHOOK set. Would have sent:")
            print(json.dumps(payload, indent=2))
        return True

    try:
        response = requests.post(
            url,
            data=json.dumps(payload),
            headers={"Content-Type": "application/json"},
            timeout=10,
        )
        response.raise_for_status()
        if verbose:
            print(f"Sent to Slack (status {response.status_code})")
        return True
    except requests.RequestException as exc:
        if verbose:
            print(f"Slack send failed: {exc}")
        return False


## Acceptance criterion

In [ ]:
from bird_watcher.send_alert import build_sighting_alert, send_alert_to_slack

jpg_files = sorted(SNAPSHOT_FOLDER.glob("*.jpg"))
snapshot_file = SNAPSHOT_FOLDER / jpg_files[0].name if jpg_files else SNAPSHOT_FOLDER / "missing.jpg"

row_id = save_sighting_to_db(DB_FILE, snapshot_file, "Northern Cardinal", 0.92, verbose=False)
payload = build_sighting_alert("Northern Cardinal", 0.92, snapshot_file, sighting_id=row_id)
sent = send_alert_to_slack(payload, webhook_url=SLACK_WEBHOOK)
assert sent, "send_alert_to_slack should return True (dry-run if no webhook)"
print("✅ Notification flow works")

## What's next

**Step 8:** open [08-web-hello.ipynb](08-web-hello.ipynb) — we'll start a tiny web app with FastAPI.